In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from anthropic import Anthropic
from dotenv import load_dotenv
from openai import OpenAI

## Settings

In [ ]:
# Pandas display options
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [ ]:
# Read .env file
load_dotenv(
    dotenv_path=Path().resolve().parent.parent / ".env",
    override=False,
)

# Set OpenAI API key
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

# Set Anthropic API key
ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]

In [ ]:
# Batch input directory
TIME_TAG = "20260222222429"
BATCH_INPUT_DIR = Path("./batch_input") / f"batch_input_{TIME_TAG}"

if not BATCH_INPUT_DIR.exists():
    raise FileNotFoundError(f"Batch input directory does not exist: {BATCH_INPUT_DIR}")

# Batch output directory
BATCH_OUTPUT_DIR = Path("./batch_output") / f"batch_output_{TIME_TAG}"
BATCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Results directory
RESULTS_DIR = Path("./results") / f"results_{TIME_TAG}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Model lists
OPENAI_NON_REASONING_MODELS = [
    "gpt-4.1-mini-2025-04-14",
    "gpt-4o-mini-2024-07-18",
]
OPENAI_REASONING_MODELS = [
    "gpt-5-mini-2025-08-07",
    "o4-mini-2025-04-16",
]
OPENAI_MODELS = OPENAI_NON_REASONING_MODELS + OPENAI_REASONING_MODELS

ANTHROPIC_MODELS = [
    "claude-sonnet-4-5-20250929",
    "claude-haiku-4-5-20251001",
    "claude-3-5-haiku-20241022",
]

## Check batch status and save results

In [ ]:
# Read batch input logs
batch_input_logs_df = pd.read_csv(BATCH_INPUT_DIR / "batch_input_logs.csv")
display(batch_input_logs_df)

In [ ]:
# Check batch status and save results
for i, batch_input_logs_row in batch_input_logs_df.iterrows():
    dataset_name = batch_input_logs_row["dataset_name"]
    model = batch_input_logs_row["model"]
    repeat = batch_input_logs_row["repeat"]
    forward_or_reverse = batch_input_logs_row["forward_or_reverse"]
    print("-" * 100)
    print(
        f"dataset: {dataset_name:<15}| model: {model:<15}| repeat: {repeat:<2}| forward_or_reverse: {forward_or_reverse:<8}"
    )

    batch_id = batch_input_logs_row["batch_id"]

    if model in OPENAI_MODELS:
        client = OpenAI(api_key=OPENAI_API_KEY)

        batch = client.batches.retrieve(batch_id)
        print(f"Batch {batch_id} processing status is {batch.status}.")

        if batch.status == "completed":
            request_counts = batch.request_counts
            print(
                f"completed: {request_counts.completed}, failed: {request_counts.failed}, total: {request_counts.total}"
            )

            # Save results
            if batch.output_file_id is not None:
                file_response = client.files.content(batch.output_file_id)
                with open(BATCH_OUTPUT_DIR / f"{batch_id}_output.jsonl", "wb") as f:
                    f.write(file_response.content)

            if batch.error_file_id is not None:
                file_response = client.files.content(batch.error_file_id)
                with open(BATCH_OUTPUT_DIR / f"{batch_id}_error.jsonl", "wb") as f:
                    f.write(file_response.content)

    elif model in ANTHROPIC_MODELS:
        client = Anthropic(api_key=ANTHROPIC_API_KEY)

        batch = client.messages.batches.retrieve(batch_id)
        print(f"Batch {batch_id} processing status is {batch.processing_status}")

        if batch.processing_status == "ended":
            request_counts = batch.request_counts
            print(f"succeeded: {request_counts.succeeded}, errored: {request_counts.errored}")

            # Save results
            results_url = batch.results_url
            headers = {"x-api-key": ANTHROPIC_API_KEY, "anthropic-version": "2023-06-01"}
            with (
                requests.get(results_url, headers=headers, stream=True) as r,
                open(BATCH_OUTPUT_DIR / f"{batch_id}_output.jsonl", "wb") as f,
            ):
                for line in r.iter_lines():
                    f.write(line + b"\n")

## Check successful counts, tokens and costs

In [ ]:
# Function to calculate cost
def calculate_cost(model, input_tokens, output_tokens, is_batch=False):
    price_USD_per_1M_tokens = {
        "gpt-5-mini-2025-08-07": {"input": 0.25, "output": 2.00},
        "o4-mini-2025-04-16": {"input": 1.10, "output": 4.40},
        "gpt-4.1-mini-2025-04-14": {"input": 0.40, "output": 1.60},
        "gpt-4o-mini-2024-07-18": {"input": 0.15, "output": 0.60},
        "claude-sonnet-4-5-20250929": {"input": 3.0, "output": 15.0},
        "claude-haiku-4-5-20251001": {"input": 1.0, "output": 5.0},
        "claude-3-5-haiku-20241022": {"input": 0.8, "output": 4.0},
    }
    cost = (
        price_USD_per_1M_tokens[model]["input"] * input_tokens
        + price_USD_per_1M_tokens[model]["output"] * output_tokens
    ) / 1000000
    if is_batch:
        cost /= 2
    return cost

In [ ]:
# Check successful counts, tokens and costs
logs = []
for i, batch_input_logs_row in batch_input_logs_df.iterrows():
    dataset_name = batch_input_logs_row["dataset_name"]
    model = batch_input_logs_row["model"]
    max_output_tokens = batch_input_logs_row["max_output_tokens"]
    reasoning_effort = batch_input_logs_row["reasoning_effort"]
    temperature = batch_input_logs_row["temperature"]
    n_samples_input = batch_input_logs_row["n_samples_input"]
    repeat = batch_input_logs_row["repeat"]
    forward_or_reverse = batch_input_logs_row["forward_or_reverse"]
    batch_id = batch_input_logs_row["batch_id"]

    # Read batch output file
    with open(BATCH_OUTPUT_DIR / f"{batch_id}_output.jsonl", "r", encoding="utf-8") as f:
        response_jsons = [json.loads(l) for l in f.readlines()]
    n_samples_output = len(response_jsons)

    # Check successful counts
    if model in OPENAI_MODELS:
        n_samples_output_succeed = np.sum(
            [True for response_json in response_jsons if response_json["response"]["status_code"] == 200]
        )
    elif model in ANTHROPIC_MODELS:
        n_samples_output_succeed = np.sum(
            [True for response_json in response_jsons if response_json["result"]["type"] == "succeeded"]
        )
    else:
        raise ValueError(f"Unsupported model: {model}")

    # Check tokens and costs
    input_tokens_list = []
    output_tokens_list = []
    cost_list = []
    for response_json in response_jsons:
        if model in OPENAI_MODELS:
            input_tokens = response_json["response"]["body"]["usage"]["input_tokens"]
            output_tokens = response_json["response"]["body"]["usage"]["output_tokens"]
        elif model in ANTHROPIC_MODELS:
            input_tokens = response_json["result"]["message"]["usage"]["input_tokens"]
            output_tokens = response_json["result"]["message"]["usage"]["output_tokens"]
        else:
            raise ValueError(f"Unsupported model: {model}")

        cost = calculate_cost(model, input_tokens, output_tokens, is_batch=True)

        input_tokens_list.append(input_tokens)
        output_tokens_list.append(output_tokens)
        cost_list.append(cost)

    logs.append(
        {
            "dataset_name": dataset_name,
            "model": model,
            "max_output_tokens": max_output_tokens,
            "reasoning_effort": reasoning_effort,
            "temperature": temperature,
            "repeat": repeat,
            "forward_or_reverse": forward_or_reverse,
            "n_samples_input": n_samples_input,
            "n_samples_output": n_samples_output,
            "n_samples_output_succeed": n_samples_output_succeed,
            "batch_id": batch_id,
            "input_tokens_mean": np.mean(input_tokens_list),
            "input_tokens_min": np.min(input_tokens_list),
            "input_tokens_max": np.max(input_tokens_list),
            "input_tokens_sum": np.sum(input_tokens_list),
            "output_tokens_mean": np.mean(output_tokens_list),
            "output_tokens_min": np.min(output_tokens_list),
            "output_tokens_max": np.max(output_tokens_list),
            "output_tokens_sum": np.sum(output_tokens_list),
            "cost_mean": np.mean(cost_list),
            "cost_sum": np.sum(cost_list),
        }
    )

# Save logs
logs_df = pd.DataFrame(logs)
display(logs_df)
logs_df.to_csv(BATCH_OUTPUT_DIR / "batch_output_logs.csv", index=False)

## Check outputs

In [ ]:
# Check outputs
for i, batch_input_logs_row in batch_input_logs_df.iterrows():
    dataset_name = batch_input_logs_row["dataset_name"]
    model = batch_input_logs_row["model"]
    repeat = batch_input_logs_row["repeat"]
    forward_or_reverse = batch_input_logs_row["forward_or_reverse"]
    batch_id = batch_input_logs_row["batch_id"]
    print("-" * 100)
    print(
        f"dataset: {dataset_name:<15}| model: {model:<15}| repeat: {repeat:<2}| forward_or_reverse: {forward_or_reverse:<8}"
    )

    # Read batch output file
    with open(BATCH_OUTPUT_DIR / f"{batch_id}_output.jsonl", "r", encoding="utf-8") as f:
        response_jsons = [json.loads(l) for l in f.readlines()]

    # Display output for each response
    for response_json in response_jsons:
        if model in OPENAI_MODELS:
            custom_id = response_json["custom_id"]
            outputs = response_json["response"]["body"]["output"]
            output_text = ""
            for output in outputs:
                if "content" in output:
                    output_text = output["content"][0]["text"]
            if not output_text:
                raise ValueError("No output text found")
        elif model in ANTHROPIC_MODELS:
            custom_id = response_json["custom_id"]
            output_text = response_json["result"]["message"]["content"][0]["input"]
        else:
            raise ValueError(f"Unsupported model: {model}")

        print(f"{custom_id:<25}: {output_text}")

## Save outputs

In [ ]:
# Save outputs
for i, batch_input_logs_row in batch_input_logs_df.iterrows():
    dataset_name = batch_input_logs_row["dataset_name"]
    model = batch_input_logs_row["model"]
    repeat = batch_input_logs_row["repeat"]
    forward_or_reverse = batch_input_logs_row["forward_or_reverse"]
    batch_id = batch_input_logs_row["batch_id"]
    print("-" * 100)
    print(
        f"dataset: {dataset_name:<15}| model: {model:<15}| repeat: {repeat:<2}| forward_or_reverse: {forward_or_reverse:<8}"
    )

    # Read batch output file
    with open(BATCH_OUTPUT_DIR / f"{batch_id}_output.jsonl", "r", encoding="utf-8") as f:
        response_jsons = [json.loads(l) for l in f.readlines()]

    # Save each response output as JSON
    output_json_list = []
    for response_json in response_jsons:
        if model in OPENAI_MODELS:
            custom_id = response_json["custom_id"]
            output_text = ""
            outputs = response_json["response"]["body"]["output"]
            for output in outputs:
                if "content" in output:
                    output_text = output["content"][0]["text"]
            if not output_text:
                raise ValueError("No output text found")
            output_json = json.loads(output_text)
        elif model in ANTHROPIC_MODELS:
            custom_id = response_json["custom_id"]
            output_text = response_json["result"]["message"]["content"][0]["input"]
            if isinstance(output_text, dict) and set(output_text.keys()) == {"answer_token", "need_more_info_token"}:
                output_json = output_text
            else:
                output_json = json.loads(output_text)
        else:
            raise ValueError(f"Unsupported model: {model}")

        output_json["ID_A"] = int(custom_id.split("-")[1].replace("A", ""))
        output_json["ID_B"] = int(custom_id.split("-")[2].replace("B", ""))
        output_json_list.append(output_json)

    # Convert output JSON to DataFrame and save
    df = pd.DataFrame(output_json_list)
    df.to_csv(
        RESULTS_DIR / f"dataset_{dataset_name}_{model}_rep{repeat}_{forward_or_reverse}_pair_result.csv", index=False
    )